In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

windows = True

if windows:
    base_dir = "P:/workspaces/lg-ukbiobank/projects/evo"
else:
    base_dir = "/data/workspaces/lag/workspaces/lg-ukbiobank/projects/evo"

fns = ["fetal_hge_hg19.merged.sorted.bed",
       "HAR_hg19.sorted.bed",
       "nean_RA_hg19-rinker_et_al.sorted.bed",
       "neanDepRegions_hg19.sorted.bed"]

col_names = ["CHR", "START", "STOP"]

def read_bed(fn, col_names):
    return pd.read_csv(fn, header=None, names=col_names, sep="\t")

def read_bed_har(fn, col_names):
    return pd.read_csv(fn, header=None, names=col_names, sep="\t", usecols=[0,1,2])

def analyze_annotation_overlaps(annotations_dict, chr_name, start_pos, end_pos, output_csv=None):
    """
    Analyze overlaps between genomic annotations and a target region.
    
    Parameters:
    -----------
    annotations_dict : dict
        Dictionary with annotation names as keys and DataFrames as values
    chr_name : str
        Chromosome name (e.g., 'chr3' or '3')
    start_pos : int
        Start position of target region
    end_pos : int  
        End position of target region
    output_csv : str, optional
        Path to save detailed overlap results as CSV
        
    Returns:
    --------
    dict : Dictionary containing overlap results for each annotation type
    """
    
    # Normalize chromosome format
    chr_formats = [chr_name, chr_name.replace('chr', ''), f"chr{chr_name.replace('chr', '')}"]
    
    overlap_results = {}
    detailed_overlaps = []
    
    for ann_name, ann_df in annotations_dict.items():
        # Filter for target chromosome
        chr_matches = ann_df['CHR'].isin(chr_formats)
        chr_ann = ann_df[chr_matches].copy()
        
        if chr_ann.empty:
            overlap_results[ann_name] = {
                'total_regions': 0,
                'overlapping_regions': 0,
                'overlap_count': 0,
                'total_coverage_bp': 0,
                'overlap_percentage': 0.0
            }
            continue
            
        # Find overlapping regions
        overlaps = chr_ann[
            (chr_ann['START'] <= end_pos) & 
            (chr_ann['STOP'] >= start_pos)
        ].copy()
        
        total_coverage = 0
        for _, row in overlaps.iterrows():
            # Calculate actual overlap coordinates
            overlap_start = max(row['START'], start_pos)
            overlap_end = min(row['STOP'], end_pos)
            overlap_length = overlap_end - overlap_start
            total_coverage += overlap_length
            
            # Classify overlap type
            if row['START'] >= start_pos and row['STOP'] <= end_pos:
                overlap_type = 'complete_containment'
            elif overlap_start == start_pos and overlap_end == end_pos:
                overlap_type = 'complete_overlap'
            else:
                overlap_type = 'partial_overlap'
            
            # Store detailed information
            detailed_overlaps.append({
                'annotation_type': ann_name,
                'chr': chr_name,
                'target_start': start_pos,
                'target_end': end_pos,
                'ann_start': row['START'],
                'ann_end': row['STOP'],
                'overlap_start': overlap_start,
                'overlap_end': overlap_end,
                'overlap_length_bp': overlap_length,
                'overlap_type': overlap_type,
                'overlap_percentage': (overlap_length / (end_pos - start_pos)) * 100
            })
        
        # Summary statistics
        target_length = end_pos - start_pos
        overlap_results[ann_name] = {
            'total_regions': len(chr_ann),
            'overlapping_regions': len(overlaps),
            'overlap_count': len(overlaps),
            'total_coverage_bp': total_coverage,
            'overlap_percentage': (total_coverage / target_length) * 100 if target_length > 0 else 0.0
        }
    
    # Save detailed results if requested
    if output_csv and detailed_overlaps:
        detailed_df = pd.DataFrame(detailed_overlaps)
        detailed_df.to_csv(output_csv, index=False)
        print(f"Detailed overlap results saved to: {output_csv}")
    
    return overlap_results, detailed_overlaps

def plot_annotation_tracks(annotations_dict, chr_name, start_pos, end_pos, lead_snp=None, 
                          sig_region=None, track_height=0.4, track_spacing=0.2, figsize=(14, 5),
                          output_formats=None, output_prefix="annotation_tracks"):
    """
    Plot genomic annotation tracks in evolutionary timeline order.
    
    Parameters:
    -----------
    annotations_dict : dict
        Dictionary with annotation names as keys and DataFrames as values
    chr_name : str
        Chromosome name
    start_pos, end_pos : int
        Plot region coordinates  
    lead_snp : int, optional
        Position of lead SNP to highlight
    sig_region : tuple, optional
        Tuple of (start, end) positions for significant region boundaries
    track_height : float
        Height of each annotation track
    track_spacing : float
        Vertical spacing between tracks
    figsize : tuple
        Figure size (width, height)
    output_formats : list
        List of formats to save ['png', 'pdf', 'svg']
    output_prefix : str
        Prefix for output files
        
    Returns:
    --------
    fig, ax : matplotlib figure and axis objects
    """
    
    # Evolutionary timeline order and colors - match the actual dictionary keys
    track_order = ['nean_intro', 'nean_dep', 'har', 'hge']
    track_colors = {
        'hge': "#80BBB4",       # Red - Human gained elements
        'har':  "#4B1742",      # Orange - Human accelerated regions  
        'nean_intro':  "#F7B682", # Blue - Neanderthal introgression
        'nean_dep':  "#ED6B06"   # Green - Neanderthal depletion
    }
    
    track_labels = {
        'hge': 'Fetal Human Gained Enhancers',
        'har': 'Human Accelerated Regions',
        'nean_intro': 'Neandertal Introgressed Regions', 
        'nean_dep': 'Neandertal Depleted Regions'
    }
    
    # Normalize chromosome format
    chr_formats = [chr_name, chr_name.replace('chr', ''), f"chr{chr_name.replace('chr', '')}"]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each annotation track
    for i, ann_name in enumerate(track_order):
        if ann_name not in annotations_dict:
            continue
            
        ann_df = annotations_dict[ann_name]
        
        # Filter for target chromosome and region
        chr_matches = ann_df['CHR'].isin(chr_formats)
        region_ann = ann_df[chr_matches & 
                          (ann_df['START'] <= end_pos) & 
                          (ann_df['STOP'] >= start_pos)].copy()
        
        # Y position for this track (bottom to top)
        y_pos = i * (track_height + track_spacing)
        
        if not region_ann.empty:
            # Plot annotation boxes
            for _, row in region_ann.iterrows():
                # Clip coordinates to plot region
                plot_start = max(row['START'], start_pos)
                plot_end = min(row['STOP'], end_pos)
                
                # Draw filled rectangle for annotation
                ax.add_patch(plt.Rectangle((plot_start, y_pos), 
                                         plot_end - plot_start, track_height,
                                         facecolor=track_colors[ann_name], 
                                         alpha=0.7, edgecolor='black', linewidth=0.5))
        
        # Draw baseline for track (even if no annotations) - align with text center
        ax.axhline(y=y_pos + track_height/2, xmin=0, xmax=1, color=track_colors[ann_name], alpha=0.3, linewidth=1)
        
        # Add track label
        ax.text(start_pos - (end_pos - start_pos) * 0.02, y_pos + track_height/2, 
               track_labels[ann_name], ha='right', va='center', fontsize=10, weight='bold')
    
    # Plot lead SNP if provided
    if lead_snp is not None and start_pos <= lead_snp <= end_pos:
        max_y = len(track_order) * (track_height + track_spacing)
        ax.axvline(x=lead_snp, color='red', linewidth=2, alpha=0.8, linestyle='--')
    
    # Plot significant region boundaries if provided
    if sig_region is not None:
        sig_start, sig_end = sig_region
        max_y = len(track_order) * (track_height + track_spacing)
        if start_pos <= sig_start <= end_pos:
            ax.axvline(x=sig_start, color='black', linewidth=1.5, alpha=0.7, linestyle=':', zorder=1)
        if start_pos <= sig_end <= end_pos:
            ax.axvline(x=sig_end, color='black', linewidth=1.5, alpha=0.7, linestyle=':', zorder=1)
    
    # Format axes
    ax.set_xlim(start_pos, end_pos)
    ax.set_ylim(-0.1, len(track_order) * (track_height + track_spacing) - track_spacing + 0.1)
    
    # X-axis formatting (Mb units)
    ax.set_xlabel(f'Position on {chr_name} (Mb)', fontsize=12, weight='bold')
    
    # Convert to Mb for tick labels
    ticks = ax.get_xticks()
    ax.set_xticklabels([f'{tick:.1f}' for tick in ticks])
    
    # Remove y-axis ticks and labels
    ax.set_yticks([])
    ax.set_ylabel('')
    
    # Title
    region_mb = (end_pos - start_pos)
    #ax.set_title(f'Evolutionary Genomic Annotations\n{chr_name}:{start_pos:,}-{end_pos:,} ({region_mb:.1f} Mb)', 
    #            fontsize=14, weight='bold', pad=20)
    
    # Grid
    ax.grid(axis='x', alpha=0.3)
    
    # Tight layout
    plt.tight_layout()
    
    # Save in multiple formats if requested
    if output_formats:
        for fmt in output_formats:
            filename = f"{output_prefix}.{fmt}"
            if fmt == 'svg':
                plt.savefig(filename, format='svg', bbox_inches='tight', dpi=300)
            elif fmt == 'pdf': 
                plt.savefig(filename, format='pdf', bbox_inches='tight', dpi=300)
            elif fmt == 'png':
                plt.savefig(filename, format='png', bbox_inches='tight', dpi=300)
            print(f"Plot saved as: {filename}")
    
    return fig, ax

def genomic_annotation_analysis(annotations_dict, chr_name, start_pos, end_pos, lead_snp=None,
                              output_dir=".", analysis_name="genomic_analysis",
                              plot_region_start=None, plot_region_end=None,
                              save_csv=True, save_plots=True, 
                              plot_formats=['png', 'pdf', 'svg'],
                              track_height=0.4, track_spacing=0.1, figsize=(14, 4)):
    """
    Complete genomic annotation analysis: overlap analysis + visualization.
    
    Parameters:
    -----------
    annotations_dict : dict
        Dictionary with annotation names as keys and DataFrames as values
    chr_name : str
        Chromosome name
    start_pos, end_pos : int
        Target region coordinates for overlap analysis
    lead_snp : int, optional
        Position of lead SNP to highlight
    output_dir : str
        Directory to save output files
    analysis_name : str
        Prefix for output files
    plot_region_start, plot_region_end : int, optional
        Region to plot (if different from analysis region)
    save_csv : bool
        Whether to save overlap results as CSV
    save_plots : bool
        Whether to save plots
    plot_formats : list
        Formats for plot output ['png', 'pdf', 'svg']
    track_height, track_spacing : float
        Plot formatting parameters
    figsize : tuple
        Figure size
        
    Returns:
    --------
    tuple : (overlap_results, detailed_overlaps, fig, ax)
    """
    
    print(f"Running genomic annotation analysis for {chr_name}:{start_pos:,}-{end_pos:,}")
    print("=" * 60)
    
    # Set plot region (default to analysis region if not specified)
    if plot_region_start is None:
        plot_region_start = start_pos
    if plot_region_end is None:
        plot_region_end = end_pos
    
    # 1. Overlap Analysis
    print("1. Analyzing overlaps with annotations...")
    
    csv_path = None
    if save_csv:
        csv_path = os.path.join(output_dir, f"{analysis_name}_overlaps.csv")
    
    overlap_results, detailed_overlaps = analyze_annotation_overlaps(
        annotations_dict, chr_name, start_pos, end_pos, output_csv=csv_path
    )
    
    # Print summary results
    print("\nOverlap Analysis Results:")
    print("-" * 40)
    
    for ann_name, results in overlap_results.items():
        print(f"\n{ann_name.upper()}:")
        print(f"  Total regions in chromosome: {results['total_regions']:,}")
        print(f"  Overlapping regions: {results['overlapping_regions']:,}")
        print(f"  Total coverage: {results['total_coverage_bp']:,} bp")
        print(f"  Coverage percentage: {results['overlap_percentage']:.2f}%")
    
    # 2. Visualization
    print(f"\n2. Creating annotation track plot...")
    print(f"   Plot region: {chr_name}:{plot_region_start:,}-{plot_region_end:,}")
    
    plot_formats_to_use = plot_formats if save_plots else None
    output_prefix = os.path.join(output_dir, analysis_name) if save_plots else None
    
    fig, ax = plot_annotation_tracks(
        annotations_dict, chr_name, plot_region_start, plot_region_end,
        lead_snp=lead_snp, sig_region=(start_pos, end_pos),
        track_height=track_height, track_spacing=track_spacing,
        figsize=figsize, output_formats=plot_formats_to_use, output_prefix=output_prefix
    )
    
    print(f"\n3. Analysis complete!")
    print(f"   Target region: {chr_name}:{start_pos:,}-{end_pos:,}")
    if lead_snp:
        print(f"   Lead SNP: {lead_snp:,}")
    print(f"   Total annotations with overlaps: {sum(1 for r in overlap_results.values() if r['overlapping_regions'] > 0)}")
    
    return overlap_results, detailed_overlaps, fig, ax

def analyze_multiple_regions(annotations_dict, regions, output_dir=".", 
                            save_combined_csv=True, save_plots=True,
                            plot_formats=['png', 'pdf', 'svg'],
                            track_height=0.4, track_spacing=0.1, figsize=(14, 4)):
    """
    Analyze multiple genomic regions together and create individual plots.
    
    Parameters:
    -----------
    annotations_dict : dict
        Dictionary with annotation names as keys and DataFrames as values
    regions : list of dict
        List of region dictionaries with keys: chr_name, start_sig, stop_sig, 
        start_plot, stop_plot, lead_snp, name
    output_dir : str
        Directory to save output files
    save_combined_csv : bool
        Whether to save combined overlap results as CSV
    save_plots : bool
        Whether to save plots
    plot_formats : list
        Formats for plot output
    track_height, track_spacing : float
        Plot formatting parameters
    figsize : tuple
        Figure size
        
    Returns:
    --------
    dict : Dictionary with region names as keys and analysis results as values
    """
    
    print("=" * 80)
    print(f"ANALYZING {len(regions)} GENOMIC REGIONS")
    print("=" * 80)
    
    all_results = {}
    all_detailed_overlaps = []
    
    # 1. Perform overlap analysis for all regions
    print("\n1. OVERLAP ANALYSIS FOR ALL REGIONS")
    print("-" * 80)
    
    for region in regions:
        region_name = region['name']
        print(f"\nAnalyzing {region_name}...")
        print(f"  Region: {region['chr_name']}:{region['start_sig']:,}-{region['stop_sig']:,}")
        
        overlap_results, detailed_overlaps = analyze_annotation_overlaps(
            annotations_dict, 
            region['chr_name'], 
            region['start_sig'], 
            region['stop_sig'],
            output_csv=None  # Don't save individual CSVs yet
        )
        
        # Add region identifier to detailed overlaps
        for overlap in detailed_overlaps:
            overlap['region_name'] = region_name
        
        all_results[region_name] = {
            'overlap_results': overlap_results,
            'detailed_overlaps': detailed_overlaps,
            'region_info': region
        }
        all_detailed_overlaps.extend(detailed_overlaps)
        
        # Print summary
        for ann_name, results in overlap_results.items():
            if results['overlapping_regions'] > 0:
                print(f"    {ann_name}: {results['overlapping_regions']} overlaps, "
                      f"{results['overlap_percentage']:.2f}% coverage")
    
    # 2. Save combined overlap results
    if save_combined_csv and all_detailed_overlaps:
        combined_csv = os.path.join(output_dir, "combined_overlaps_all_regions.csv")
        detailed_df = pd.DataFrame(all_detailed_overlaps)
        detailed_df.to_csv(combined_csv, index=False)
        print(f"\n2. Combined overlap results saved to: {combined_csv}")
    
    # 3. Create individual plots for each region
    print(f"\n3. CREATING INDIVIDUAL PLOTS FOR EACH REGION")
    print("-" * 80)
    
    figs = {}
    for region in regions:
        region_name = region['name']
        print(f"\nPlotting {region_name}...")
        print(f"  Plot region: {region['chr_name']}:{region['start_plot']:,}-{region['stop_plot']:,}")
        
        plot_formats_to_use = plot_formats if save_plots else None
        output_prefix = os.path.join(output_dir, f"{region_name}_tracks") if save_plots else None
        
        fig, ax = plot_annotation_tracks(
            annotations_dict, 
            region['chr_name'], 
            region['start_plot'], 
            region['stop_plot'],
            lead_snp=region.get('lead_snp'),
            sig_region=(region['start_sig'], region['stop_sig']),
            track_height=track_height, 
            track_spacing=track_spacing,
            figsize=figsize, 
            output_formats=plot_formats_to_use, 
            output_prefix=output_prefix
        )
        
        figs[region_name] = (fig, ax)
    
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE!")
    print(f"Total regions analyzed: {len(regions)}")
    print(f"Total overlaps found: {len(all_detailed_overlaps)}")
    print("=" * 80)
    
    return all_results, figs

In [2]:
hge = read_bed(os.path.join(base_path, fns[0]), col_names)
har = read_bed_har(os.path.join(base_path, fns[1]), col_names)
nean_intro = read_bed(os.path.join(base_path, fns[2]), col_names)
nean_dep = read_bed(os.path.join(base_path, fns[3]), col_names)

# Create annotations dictionary for the loaded data
annotations = {
    'hge': hge,
    'har': har, 
    'nean_intro': nean_intro,
    'nean_dep': nean_dep
}

In [ ]:
# Define all regions to analyze
regions = [{
        'name': 'chr2_NRXN1_sig_region',
        'chr_name': 'chr2',
        'start_sig': 50535715,
        'stop_sig': 50549465,
        'start_plot': 50400000,
        'stop_plot': 50800000,
        'lead_snp': 50549465
        },
    
    {
        'name': 'chr10_PLCE1_sig_region',
        'chr_name': 'chr10',
        'start_sig': 95977011,
        'stop_sig': 96133084,
        'start_plot': 95700000,
        'stop_plot': 96300000,
        'lead_snp': 96015793
    },
    {
        'name': 'chr10_INPP5A_sig_region',
        'chr_name': 'chr10',
        'start_sig': 134269304,
        'stop_sig': 134335986,
        'start_plot': 134200000,
        'stop_plot': 134500000,
        'lead_snp': 134297801
    },

    
]

# Run combined analysis with individual plots
all_results, figs = analyze_multiple_regions(
    annotations_dict=annotations,
    regions=regions,
    output_dir=base_dir,
    save_combined_csv=True,
    save_plots=True,
    plot_formats=['png', 'pdf', 'svg'],
    track_height=0.4,
    track_spacing=0.1,
    figsize=(14, 4)
)

# Display all plots
for region_name, (fig, ax) in figs.items():
    plt.figure(fig.number)
    plt.show()

In [4]:
#OLD
""" 
regions = [
        {
        'name': 'chr2_sig_region',
        'chr_name': 'chr2',
        'start_sig': 114065572,
        'stop_sig': 114110568,
        'start_plot': 114000000,
        'stop_plot': 114200000,
        'lead_snp': 114103966
        },
    {
        'name': 'chr3_sig_region',
        'chr_name': 'chr3',
        'start_sig': 89451721,
        'stop_sig': 89751353,
        'start_plot': 89200000,
        'stop_plot': 90000000,
        'lead_snp': 89521693
    },
    {
        'name': 'chr10_sig_region',
        'chr_name': 'chr10',
        'start_sig': 96009182,
        'stop_sig': 96069405,
        'start_plot': 95700000,
        'stop_plot': 96300000,
        'lead_snp': 96012950
    },
    {
        'name': 'chr15_sig_region',
        'chr_name': 'chr15',
        'start_sig': 91424574,
        'stop_sig': 91424574,
        'start_plot': 91400000,
        'stop_plot': 91460000,
        'lead_snp': 91424574
    },
    {
        'name': 'chr22_sig_region',
        'chr_name': 'chr22',
        'start_sig': 47164865,
        'stop_sig': 47235159,
        'start_plot': 47000000,
        'stop_plot': 47400000,
        'lead_snp': 47190735
    }
] """

" \nregions = [\n        {\n        'name': 'chr2_sig_region',\n        'chr_name': 'chr2',\n        'start_sig': 114065572,\n        'stop_sig': 114110568,\n        'start_plot': 114000000,\n        'stop_plot': 114200000,\n        'lead_snp': 114103966\n        },\n    {\n        'name': 'chr3_sig_region',\n        'chr_name': 'chr3',\n        'start_sig': 89451721,\n        'stop_sig': 89751353,\n        'start_plot': 89200000,\n        'stop_plot': 90000000,\n        'lead_snp': 89521693\n    },\n    {\n        'name': 'chr10_sig_region',\n        'chr_name': 'chr10',\n        'start_sig': 96009182,\n        'stop_sig': 96069405,\n        'start_plot': 95700000,\n        'stop_plot': 96300000,\n        'lead_snp': 96012950\n    },\n    {\n        'name': 'chr15_sig_region',\n        'chr_name': 'chr15',\n        'start_sig': 91424574,\n        'stop_sig': 91424574,\n        'start_plot': 91400000,\n        'stop_plot': 91460000,\n        'lead_snp': 91424574\n    },\n    {\n       

In [5]:
# Optional: Access individual results
for region_name, result in all_results.items():
    print(f"\n{region_name.upper()}:")
    print("-" * 40)
    for ann_name, overlap in result['overlap_results'].items():
        if overlap['overlapping_regions'] > 0:
            print(f"{ann_name}: {overlap['overlapping_regions']} overlaps, "
                  f"{overlap['total_coverage_bp']:,} bp, "
                  f"{overlap['overlap_percentage']:.2f}% coverage")


CHR2_NRXN1_SIG_REGION:
----------------------------------------

CHR10_PLCE1_SIG_REGION:
----------------------------------------
nean_intro: 26 overlaps, 26 bp, 0.02% coverage

CHR10_INPP5A_SIG_REGION:
----------------------------------------
hge: 3 overlaps, 17,036 bp, 25.55% coverage
nean_intro: 11 overlaps, 11 bp, 0.02% coverage
